In [13]:
from tqdm import tqdm
import pandas as pd
import spacy
import re
import unicodedata
from unidecode import unidecode
import ast
from collections import Counter

In [14]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
ents = pd.read_csv("/Users/pol/Downloads/extracted_ents_ALL.csv", sep=';')
ents = ents.dropna(subset=["name"])
ents

press = pd.read_csv("/Users/pol/Downloads/agg_matches_ALL.csv", sep=';')
press

,article_id,article_text,name,name_count
0,1,Serge Reggiani - Succès & confidences .\n Une ...,"['Serge Reggiani - Succès & confidences', 'Reg...",1
1,2,"Désolés, Warren .\n C'est une maigre consolati...","['Warren', 'Warren Zevon', ""I'll sleep when I'...",10
2,3,Adam Green - Friends of mine .\n Dire qu'on a ...,"['Adam Green - Friends of mine', 'The Kills', ...",1
3,4,Cesaria Evora - Voz d'amor .\n Dès les premièr...,"['gorgée', 'Voz', ""Voix de l'amour"", 'Velocida...",2
4,5,Herman Düne - Mas cambios et Mash Concrete Met...,"['Herman Düne - Mas cambios', 'Mash Concrete M...",1
...,...,...,...,...
52639,52690,"Rap, jazz, rock, classique, chanson… Les album...","['Rap', 'Monde', 'An Evening with Silk Sonic',...",45
52640,52691,Que sait-on vraiment des surrisques de contami...,"['Covid-19', 'Omicron', 'Jean Castex', 'Covid-...",482
52641,52692,"Rencontre avec Riccardo Muti, maestro à vie .\...","['Riccardo Muti', 'Riccardo Muti', 'Cristina',...",70
52642,52693,Ils sont passés chez Colors : la sélection mus...,"['Colors', 'Monde Afrique', 'Le Monde Afrique'...",27


### check texts with possible false positives

In [70]:
count = 0

for i, (text, namelist) in enumerate(
    zip(press["article_text"],
        press["name"]
        )):
    if ("'Johnny Cash'" in namelist and "'Johnny'"):
        #print(i)
        count += 1

print(count)

220


In [7]:
press["name"] = press["name"].apply(ast.literal_eval)
press["name"]

0        [Serge Reggiani - Succès & confidences, Reggia...
1        [Warren, Warren Zevon, I'll sleep when I'm dea...
2        [Adam Green - Friends of mine, The Kills, Gree...
3        [gorgée, Voz, Voix de l'amour, Velocidade, Ces...
4        [Herman Düne - Mas cambios, Mash Concrete Meta...
                               ...                        
52639    [Rap, Monde, An Evening with Silk Sonic, Silk ...
52640    [Covid-19, Omicron, Jean Castex, Covid-19, Clu...
52641    [Riccardo Muti, Riccardo Muti, Cristina, Ricca...
52642    [Colors, Monde Afrique, Le Monde Afrique, Colo...
52643    [Pierre-Emmanuel Barbier, Thomson, Diapason, P...
Name: name, Length: 52644, dtype: object

In [71]:
def count_name_cooc(target_name, position="first"):
    """
    namelists: iterable of lists of extracted names
    target_name: string (e.g. "christophe" or "mae")
    position: "first" or "last"
    """

    target_name = target_name.lower()
    alone_count = 0
    cooccurrence_counter = Counter()

    for namelist in press["name"]:

        names_lower = [n.lower() for n in namelist]

        # Case 1: FIRST NAME logic
        if position == "first":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.startswith(target_name + " ")]

            if has_alone and not full_names:
                alone_count += 1

            elif has_alone and full_names:
                for full_name in full_names:
                    last_part = full_name[len(target_name)+1:]
                    cooccurrence_counter[last_part] += 1

        # Case 2: LAST NAME logic
        elif position == "last":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.endswith(" " + target_name)]

            if has_alone and not full_names:
                alone_count += 1

            elif has_alone and full_names:
                for full_name in full_names:
                    first_part = full_name[:-(len(target_name)+1)]
                    cooccurrence_counter[first_part] += 1

        else:
            raise ValueError("position must be 'first' or 'last'")

    return alone_count, cooccurrence_counter

In [24]:
from collections import Counter
import re

def count_name_cooc(target_name, position="FIRST", window=10, mode="words"):
    """
    target_name: string (e.g. "jackson")
    position: "first" or "last"
    window: number of words before/after (used if mode="words")
    mode: "words" or "sentence"
    """

    target_name = target_name.lower()
    alone_count = 0
    cooccurrence_counter = Counter()
    contexts = []  # list of tuples: (article_id, context)

    for article_id, namelist, article_text in zip(
        press["article_id"], 
        press["name"], 
        press["article_text"]
    ):

        names_lower = [n.lower() for n in namelist]
        text_lower = article_text.lower()

        # -------- Determine logic --------
        if position == "FIRST":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.startswith(target_name + " ")]

        elif position == "LAST":
            has_alone = target_name in names_lower
            full_names = [n for n in names_lower if n.endswith(" " + target_name)]

        else:
            raise ValueError("position must be 'first' or 'last'")

        # -------- CASE 1: NAME ALONE --------
        if has_alone and not full_names:
            alone_count += 1

            context_found = None

            if mode == "sentence":
                sentences = re.split(r'(?<=[.!?])\s+', article_text)
                for s in sentences:
                    if target_name in s.lower():
                        context_found = s.strip()
                        break  # only first sentence

            elif mode == "words":
                words = article_text.split()
                words_lower = [re.sub(r'\W+', '', w.lower()) for w in words]

                for i, w in enumerate(words_lower):
                    if w == target_name:
                        start = max(0, i - window)
                        end = min(len(words), i + window + 1)
                        context_found = " ".join(words[start:end])
                        break  # only first match

            if context_found:
                # Highlight name
                context_found = re.sub(
                    rf"\b({target_name})\b",
                    lambda m: m.group(1).upper(),
                    context_found,
                    flags=re.IGNORECASE
                )

                contexts.append((article_id, context_found))

        # -------- CASE 2: NAME WITH OTHER PART --------
        elif has_alone and full_names:
            for full_name in full_names:
                if position == "first":
                    last_part = full_name[len(target_name)+1:]
                    cooccurrence_counter[last_part] += 1
                else:
                    first_part = full_name[:-(len(target_name)+1)]
                    cooccurrence_counter[first_part] += 1

    return alone_count, cooccurrence_counter, contexts

In [49]:
target_name = "VICE"

alone, with_last, contexts = count_name_cooc(
    target_name=target_name,
    position="LAST",
    window=24,
    mode="words"
)

print("Alone count:", alone)
print("\nTop co-occurring names:\n")

for k, v in with_last.most_common(10):
    print(f"{k}: {v}")

print("\nContexts:\n")

for article_id, snippet in contexts:
    print(f"[Article {article_id}]")
    print(snippet)
    print()

Alone count: 53

Top co-occurring names:


Contexts:

[Article 8206]
cinéma, comment avez-vous réussi à le convaincre de revenir jouer pour vous ? C'est grâce à Paul Thomas Anderson avec qui il tourne actuellement VICE caché. Il a été très persuasif et a réussi à le convaincre. Je savais que Joaquin était dans une mauvaise passe et qu'il voulait

[Article 10159]
tout de noir vêtu, avec un col montant blanc (Niels Arestrup) et Fouché, dans un bel habit à broderies d'argent (costumes Nathalie Bérard-Benoin). Le VICE et le crime Jean-Claude Brisville, qui avait déjà composé L'Entretien entre M. Descartes et M. Pascal le jeune, reprenait cette dramaturgie pour Le Souper

[Article 14813]
Gary Hart, la débandade . Pour jouer Dick Cheney dans VICE (en salle le 20 février), Christian Bale a pris du poids et s'est fait une tête de chauve. Hugh Jackman, lui, n'a besoin que

[Article 14948]
Paris, on découvre le réalisateur de The Big Short, formé au théâtre d'improvisation à Chicago, allongé sur

In [57]:
count_name_cooc(target_name=target_name,
    position="first")

(0, Counter(), [])

In [103]:
n = 10                       
count = 0
artist = "ziggy stardust"

for i, (text, namelist) in enumerate(zip(press["article_text"],
                                         press["name"])):
    if artist in namelist:
        print(i)
        print(namelist)
        print("-------------------")
        print(text)
        print("===================")

        count += 1

print(f"{count} OCCURRENCES OF '{artist}'")

0 OCCURRENCES OF 'ziggy stardust'


In [ ]:
n = 10                       
count = 0
artist = "ziggy stardust"

for i, (text, namelist) in enumerate(zip(press["article_text"],
                                         press["name"])):
    if artist in namelist:
        # split the article into tokens and look for the artist token(s)
        words = text.split()
        for idx, w in enumerate(words):
            if w.lower() == artist.lower():
                lo = max(0, idx - n)
                hi = idx + n + 1
                snippet = " ".join(words[lo:hi])
                print("------------------------")
                print(i)
                print(namelist)
                print(snippet) # only n words before/after
        count += 1

print("==================")
print(f"{count} OCCURRENCES OF '{artist}'")

0 OCCURRENCES OF ''ziggy stardust''


In [76]:
df = ents
len(df)

408677

In [98]:
df = ents

# normalize both texts
def clean_text(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = unidecode(text) # remove accents
    # text = text.replace("\xa0", " ")
    text = text.strip()
    return text


df = df[df["name"].map(len) > 2] # short names

df["name"] = df["name"].apply(clean_text)

### check names with many tokens

maybe leave for now, if we do strict matching they'll be dropped anyway

In [103]:
df = df.drop_duplicates(subset=["name"])
df.reset_index()

,index,name_id,name,name_count,article_id
0,2,3,liberation,3336,870
1,3,4,figaro,2749,1786
2,4,5,internet,2376,40
3,5,6,mozart,1831,87
4,6,7,dieu,1616,592
...,...,...,...,...,...
386005,408671,408672,i'm changing,1,49038
386006,408672,408673,i'm blind,1,48689
386007,408673,408674,i'm amazing,1,49542
386008,408674,408675,i'm all smiles,1,50227


In [105]:
df.to_csv("/Users/pol/Downloads/extracted_ents_clean.csv", sep=';')

In [ ]:
# add french stopwords from spaCy
nlp = spacy.load("fr_core_news_lg", exclude=['tagger', 'parser', 'ner', 
                                            'attribute_ruler', 'lemmatizer']) # load the small French language model

french_stopwords = nlp.Defaults.stop_words # access stopwords list
french_stopwords = list(french_stopwords) # convert to list

In [44]:
namelist = [name.lower() for name in press["name"]]
namelist

["['serge reggiani - succès & confidences', 'reggiani', 'brel', 'barbara', 'ventura', 'dernier courrier', 'cd prod', 'jacques canetti', 'polydor', 'serge reggiani']",
 '[\'warren\', \'warren zevon\', "i\'ll sleep when i\'m dead", \'roland the headless\', \'thompson gunner\', \'lawyers\', \'guns and money\', "mohammed\'s radio", \'sentimental hygiene\', \'elton john us\', \'springsteen\', \'jackson browne\', \'r.e.m.\', \'hunter thompson\', \'tom mcguane\', \'werewolves of london\', \'scorsese\', \'tom cruise\', "la couleur de l\'argent", \'mutineer\', "hugo cassavetti \\n i\'ll sleep when i\'m dead", \'cd\', \'rhino\', \'the wind\', \'naïve\']',
 '[\'adam green - friends of mine\', \'the kills\', \'green\', \'moldy peaches\', \'friends of mine\', \'bungee\', \'jonathan richman\', \'lou reed\', \'charles trenet\', "rubik\'", \'françois gorin\', \'cd pias\']',
 '[\'gorgée\', \'voz\', "voix de l\'amour", \'velocidade\', \'cesaria\', \'luis morais\', \'verte campagne\', \'greenfields\', \'

In [ ]:
for row in press["name"]:
    for name in row:
        if name not in french_stopwords:
            name
        else:
            

stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
stopword
s

KeyboardInterrupt: 